# Home Credit: Andy's 200 features

Andy's [feature pipeline](https://github.com/AndyChhuon/HomeCreditDefaultRisk/blob/48aeb521f56b8467ab7c52fcbc0319fa95bed021/home_credit.ipynb) on the fixed 100k / 25k split. The comparison uses the same model settings as `HomeCredit_Team_200.ipynb`.

## 1. Get the data

Use the course ZIP link, a local `home-credit-default-risk.zip`, or CSVs in `data/`.

In [1]:
import gc
import hashlib
import os
import shutil
from getpass import getpass
from pathlib import Path
from urllib.request import Request, urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_limits

SEED = 42
ID = 'SK_ID_CURR'
DATA_DIR = Path(os.getenv('HCD_DATA_DIR', 'data'))
ZIP_PATH = Path(os.getenv('HCD_ARCHIVE_PATH', 'home-credit-default-risk.zip'))
OUTPUT_DIR = Path('outputs_andy_200')
FILES = [
    'application_train.csv', 'application_test.csv', 'bureau.csv',
    'previous_application.csv', 'POS_CASH_balance.csv', 'installments_payments.csv',
    'credit_card_balance.csv', 'sample_submission.csv',
]

def get_data():
    if all((DATA_DIR / name).is_file() for name in FILES):
        return
    archive = ZIP_PATH
    if not archive.is_file():
        url = os.getenv('HCD_DATA_URL') or getpass('Course data ZIP URL: ')
        if not url.strip():
            raise ValueError('Provide the course ZIP URL or set ZIP_PATH to a local copy.')
        ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
        archive = ZIP_PATH.with_suffix('.zip.part')
        request = Request(url.strip(), headers={'User-Agent': 'Mozilla/5.0'})
        with urlopen(request, timeout=90) as src, archive.open('wb') as out:
            shutil.copyfileobj(src, out, length=4 << 20)
    with archive.open('rb') as f:
        digest = hashlib.file_digest(f, 'sha256').hexdigest()
    if digest != '4e7a243b13f6e3d40f682a9a0cfd59c70910ca6c5a4289f71ca1f25efbd8e3de':
        raise ValueError('The ZIP does not match the course archive.')
    if archive != ZIP_PATH:
        archive.replace(ZIP_PATH)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with ZipFile(ZIP_PATH) as z:
        for name in FILES:
            destination = DATA_DIR / name
            part = destination.with_suffix('.csv.part')
            with z.open(name) as src, part.open('wb') as out:
                shutil.copyfileobj(src, out, length=4 << 20)
            part.replace(destination)

get_data()
print('Data ready.')

Data ready.


## 2. Split the data and build history features

In [2]:
app = pd.read_csv(DATA_DIR / 'application_train.csv')
test = pd.read_csv(DATA_DIR / 'application_test.csv')
target = app['TARGET'].to_numpy(dtype=np.int32)
train_pos, val_pos = train_test_split(
    np.arange(len(app)), test_size=0.2, stratify=target, random_state=SEED,
)
train_pos, _ = train_test_split(
    train_pos, train_size=100_000, stratify=target[train_pos], random_state=SEED,
)
val_pos, _ = train_test_split(
    val_pos, train_size=25_000, stratify=target[val_pos], random_state=SEED,
)
train_ids = pd.Index(app.iloc[train_pos][ID])
val_ids = pd.Index(app.iloc[val_pos][ID])
test_ids = pd.Index(test[ID])
y_train, y_val = target[train_pos], target[val_pos]
split_key = np.r_[train_ids, val_ids].astype('<i8').tobytes()
assert hashlib.sha256(split_key).hexdigest() == '7d4dbd29f899d6163074681e4bc6f9c6fb2c47b91150b92bb7f6ce9ac0062f86'


train_apps, val_apps = app.iloc[train_pos].copy(), app.iloc[val_pos].copy()
all_ids = train_ids.append(val_ids).append(test_ids)
assert all_ids.is_unique
categorical_cols = app.select_dtypes(exclude='number').columns.tolist()
del app, target

ID_COL, TARGET = ID, 'TARGET'
EMPLOYMENT_COL, employed_sentinel = 'DAYS_EMPLOYED', 365243
PLACEHOLDER_TOKENS = ['XNA', 'XAP', 'Unknown']
print(f'Train: {len(train_ids):,}; validation: {len(val_ids):,}; test: {len(test_ids):,}.')

def read_history(name, columns):
    chunks = pd.read_csv(DATA_DIR / f'{name}.csv', usecols=[ID] + columns, chunksize=500_000)
    return pd.concat((chunk.loc[chunk[ID].isin(all_ids)] for chunk in chunks), ignore_index=True)

def aggregate(table, prefix, **features):
    """Summarise a table to one row per applicant; each keyword is feature_name=(column, statistic)."""
    return table.groupby(ID_COL).agg(**features).add_prefix(f"{prefix}_")


def ratio(numerator, denominator):
    """Divide two columns, returning NaN instead of infinity where the denominator is 0."""
    return numerator / denominator.replace(0, np.nan)

Train: 100,000; validation: 25,000; test: 48,744.


In [3]:
raw_bureau = read_history('bureau', [
    'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM',
    'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT',
])
bureau = raw_bureau
is_active = bureau["CREDIT_ACTIVE"] == "Active"
bureau = bureau.assign(
    IS_ACTIVE=is_active,
    ACTIVE_DEBT=bureau["AMT_CREDIT_SUM_DEBT"].where(is_active),
    ACTIVE_CREDIT=bureau["AMT_CREDIT_SUM"].where(is_active),
)
bureau_features = aggregate(
    bureau, "BUREAU",
    LOAN_COUNT=("SK_ID_BUREAU", "count"),
    ACTIVE_SHARE=("IS_ACTIVE", "mean"),
    ACTIVE_DEBT=("ACTIVE_DEBT", "sum"),
    ACTIVE_CREDIT=("ACTIVE_CREDIT", "sum"),
    DAYS_OVERDUE_MAX=("CREDIT_DAY_OVERDUE", "max"),
    DAYS_CREDIT_LATEST=("DAYS_CREDIT", "max"),
)
bureau_features["BUREAU_DEBT_TO_CREDIT"] = ratio(bureau_features["BUREAU_ACTIVE_DEBT"], bureau_features["BUREAU_ACTIVE_CREDIT"])
bureau_features = bureau_features.drop(columns="BUREAU_ACTIVE_CREDIT")
del raw_bureau, bureau

raw_previous = read_history('previous_application', [
    'SK_ID_PREV', 'DAYS_DECISION', 'NAME_CONTRACT_STATUS', 'AMT_CREDIT',
    'AMT_APPLICATION',
])
previous = raw_previous.sort_values(["DAYS_DECISION", "SK_ID_PREV"])
previous = previous.assign(
    IS_APPROVED=(previous["NAME_CONTRACT_STATUS"] == "Approved").astype(int),
    IS_REFUSED=(previous["NAME_CONTRACT_STATUS"] == "Refused").astype(int),
    CREDIT_TO_REQUESTED=ratio(previous["AMT_CREDIT"], previous["AMT_APPLICATION"]),
)
previous_features = aggregate(
    previous, "PREV",
    APPLICATION_COUNT=("SK_ID_PREV", "count"),
    APPROVED_SHARE=("IS_APPROVED", "mean"),
    REFUSED_SHARE=("IS_REFUSED", "mean"),
    LATEST_REFUSED=("IS_REFUSED", "last"),
    DAYS_DECISION_LATEST=("DAYS_DECISION", "max"),
    CREDIT_TO_REQUESTED_MEAN=("CREDIT_TO_REQUESTED", "mean"),
)
del raw_previous, previous

raw_pos = read_history('POS_CASH_balance', ['SK_ID_PREV', 'MONTHS_BALANCE', 'SK_DPD_DEF', 'NAME_CONTRACT_STATUS'])
pos = raw_pos.sort_values(["MONTHS_BALANCE", "SK_ID_PREV"])
pos = pos.assign(
    IS_LATE=pos["SK_DPD_DEF"] > 0,
    IS_ACTIVE=(pos["NAME_CONTRACT_STATUS"] == "Active").astype(int),
)
pos_features = aggregate(
    pos, "POS",
    LATE_MONTH_SHARE=("IS_LATE", "mean"),
    DPD_MAX=("SK_DPD_DEF", "max"),
    LATEST_ACTIVE=("IS_ACTIVE", "last"),
)
del raw_pos, pos

raw_instalments = read_history('installments_payments', [
    'SK_ID_PREV', 'NUM_INSTALMENT_VERSION', 'NUM_INSTALMENT_NUMBER', 'AMT_INSTALMENT',
    'AMT_PAYMENT', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT',
])
PAYMENT_TOLERANCE = 1.0

instalments = raw_instalments.groupby(
    [ID_COL, "SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER"], as_index=False,
).agg(
    AMT_DUE=("AMT_INSTALMENT", "first"),
    AMT_PAID=("AMT_PAYMENT", "sum"),
    DAYS_DUE=("DAYS_INSTALMENT", "first"),
    DAYS_PAID=("DAYS_ENTRY_PAYMENT", "max"),
)
days_late = (instalments["DAYS_PAID"] - instalments["DAYS_DUE"]).clip(lower=0)
instalments = instalments.assign(
    DAYS_LATE=days_late,
    IS_LATE=days_late > 0,
    IS_UNDERPAID=instalments["AMT_DUE"] - instalments["AMT_PAID"] > PAYMENT_TOLERANCE,
    PAID_TO_DUE=ratio(instalments["AMT_PAID"], instalments["AMT_DUE"]),
)
instalment_features = aggregate(
    instalments, "INST",
    LATE_SHARE=("IS_LATE", "mean"),
    DAYS_LATE_MEAN=("DAYS_LATE", "mean"),
    DAYS_LATE_MAX=("DAYS_LATE", "max"),
    UNDERPAID_SHARE=("IS_UNDERPAID", "mean"),
    PAID_TO_DUE_MEAN=("PAID_TO_DUE", "mean"),
)
del raw_instalments, instalments

raw_card = read_history('credit_card_balance', [
    'SK_ID_PREV', 'MONTHS_BALANCE', 'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL',
    'SK_DPD_DEF', 'AMT_DRAWINGS_CURRENT',
])
card = raw_card.sort_values(["MONTHS_BALANCE", "SK_ID_PREV"])
card = card.assign(
    UTILISATION=ratio(card["AMT_BALANCE"], card["AMT_CREDIT_LIMIT_ACTUAL"]),
    IS_LATE=card["SK_DPD_DEF"] > 0,
)
card_features = aggregate(
    card, "CC",
    UTILISATION_LATEST=("UTILISATION", "last"),
    UTILISATION_MEAN=("UTILISATION", "mean"),
    DRAWINGS_MEAN=("AMT_DRAWINGS_CURRENT", "mean"),
    DPD_MAX=("SK_DPD_DEF", "max"),
    LATE_MONTH_SHARE=("IS_LATE", "mean"),
)
del raw_card, card

In [4]:
aux_feature_tables = [bureau_features, previous_features, pos_features, instalment_features, card_features]
COUNT_FEATURES = ["BUREAU_LOAN_COUNT", "PREV_APPLICATION_COUNT"]


def join_aux_features(applications):
    """Left join the auxiliary features, keeping exactly one row per application."""
    joined = applications
    for features in aux_feature_tables:
        joined = joined.merge(features, left_on=ID_COL, right_index=True, how="left", validate="one_to_one")

    # No records means a count of 0; other features stay NaN for the imputers
    joined[COUNT_FEATURES] = joined[COUNT_FEATURES].fillna(0)
    joined["HAS_HC_CREDIT_CARD"] = joined[ID_COL].isin(card_features.index).astype(int)

    assert len(joined) == len(applications), "The join must not add or remove applications"
    return joined


train_sample = join_aux_features(train_apps)
val_sample = join_aux_features(val_apps)
test_joined = join_aux_features(test)
for part, ids in zip((train_sample, val_sample, test_joined), (train_ids, val_ids, test_ids)):
    assert np.array_equal(part[ID], ids)
assert np.array_equal(train_sample[TARGET], y_train) and np.array_equal(val_sample[TARGET], y_val)
del train_apps, val_apps, test, aux_feature_tables
del bureau_features, previous_features, pos_features, instalment_features, card_features
gc.collect()

## 3. Preprocess and keep Andy's 200 columns

Fixed list from Andy's `StandardScaler` + `SelectKBest(200)` run, checked against the saved 200-column list. The preprocessing below fits on the training rows.

In [5]:
ANDY_FEATURES_200 = [
    'continuous__CNT_CHILDREN',
    'continuous__AMT_INCOME_TOTAL',
    'continuous__AMT_CREDIT',
    'continuous__AMT_ANNUITY',
    'continuous__AMT_GOODS_PRICE',
    'continuous__REGION_POPULATION_RELATIVE',
    'continuous__DAYS_BIRTH',
    'continuous__DAYS_EMPLOYED',
    'continuous__DAYS_REGISTRATION',
    'continuous__DAYS_ID_PUBLISH',
    'continuous__OWN_CAR_AGE',
    'continuous__REGION_RATING_CLIENT',
    'continuous__REGION_RATING_CLIENT_W_CITY',
    'continuous__HOUR_APPR_PROCESS_START',
    'continuous__EXT_SOURCE_1',
    'continuous__EXT_SOURCE_2',
    'continuous__EXT_SOURCE_3',
    'continuous__APARTMENTS_AVG',
    'continuous__BASEMENTAREA_AVG',
    'continuous__YEARS_BUILD_AVG',
    'continuous__COMMONAREA_AVG',
    'continuous__ELEVATORS_AVG',
    'continuous__ENTRANCES_AVG',
    'continuous__FLOORSMAX_AVG',
    'continuous__FLOORSMIN_AVG',
    'continuous__LANDAREA_AVG',
    'continuous__LIVINGAPARTMENTS_AVG',
    'continuous__LIVINGAREA_AVG',
    'continuous__NONLIVINGAREA_AVG',
    'continuous__APARTMENTS_MODE',
    'continuous__BASEMENTAREA_MODE',
    'continuous__YEARS_BUILD_MODE',
    'continuous__ELEVATORS_MODE',
    'continuous__ENTRANCES_MODE',
    'continuous__FLOORSMAX_MODE',
    'continuous__FLOORSMIN_MODE',
    'continuous__LANDAREA_MODE',
    'continuous__LIVINGAPARTMENTS_MODE',
    'continuous__LIVINGAREA_MODE',
    'continuous__NONLIVINGAREA_MODE',
    'continuous__APARTMENTS_MEDI',
    'continuous__BASEMENTAREA_MEDI',
    'continuous__YEARS_BUILD_MEDI',
    'continuous__COMMONAREA_MEDI',
    'continuous__ELEVATORS_MEDI',
    'continuous__ENTRANCES_MEDI',
    'continuous__FLOORSMAX_MEDI',
    'continuous__FLOORSMIN_MEDI',
    'continuous__LANDAREA_MEDI',
    'continuous__LIVINGAPARTMENTS_MEDI',
    'continuous__LIVINGAREA_MEDI',
    'continuous__NONLIVINGAREA_MEDI',
    'continuous__TOTALAREA_MODE',
    'continuous__OBS_30_CNT_SOCIAL_CIRCLE',
    'continuous__DEF_30_CNT_SOCIAL_CIRCLE',
    'continuous__OBS_60_CNT_SOCIAL_CIRCLE',
    'continuous__DEF_60_CNT_SOCIAL_CIRCLE',
    'continuous__DAYS_LAST_PHONE_CHANGE',
    'continuous__AMT_REQ_CREDIT_BUREAU_MON',
    'continuous__AMT_REQ_CREDIT_BUREAU_YEAR',
    'continuous__BUREAU_LOAN_COUNT',
    'continuous__BUREAU_ACTIVE_SHARE',
    'continuous__BUREAU_DAYS_CREDIT_LATEST',
    'continuous__BUREAU_DEBT_TO_CREDIT',
    'continuous__PREV_APPLICATION_COUNT',
    'continuous__PREV_APPROVED_SHARE',
    'continuous__PREV_REFUSED_SHARE',
    'continuous__PREV_DAYS_DECISION_LATEST',
    'continuous__PREV_CREDIT_TO_REQUESTED_MEAN',
    'continuous__POS_LATE_MONTH_SHARE',
    'continuous__INST_LATE_SHARE',
    'continuous__INST_UNDERPAID_SHARE',
    'continuous__missingindicator_DAYS_EMPLOYED',
    'continuous__missingindicator_OWN_CAR_AGE',
    'continuous__missingindicator_EXT_SOURCE_1',
    'continuous__missingindicator_EXT_SOURCE_3',
    'continuous__missingindicator_APARTMENTS_AVG',
    'continuous__missingindicator_BASEMENTAREA_AVG',
    'continuous__missingindicator_YEARS_BEGINEXPLUATATION_AVG',
    'continuous__missingindicator_YEARS_BUILD_AVG',
    'continuous__missingindicator_COMMONAREA_AVG',
    'continuous__missingindicator_ELEVATORS_AVG',
    'continuous__missingindicator_ENTRANCES_AVG',
    'continuous__missingindicator_FLOORSMAX_AVG',
    'continuous__missingindicator_FLOORSMIN_AVG',
    'continuous__missingindicator_LANDAREA_AVG',
    'continuous__missingindicator_LIVINGAPARTMENTS_AVG',
    'continuous__missingindicator_LIVINGAREA_AVG',
    'continuous__missingindicator_NONLIVINGAPARTMENTS_AVG',
    'continuous__missingindicator_NONLIVINGAREA_AVG',
    'continuous__missingindicator_APARTMENTS_MODE',
    'continuous__missingindicator_BASEMENTAREA_MODE',
    'continuous__missingindicator_YEARS_BEGINEXPLUATATION_MODE',
    'continuous__missingindicator_YEARS_BUILD_MODE',
    'continuous__missingindicator_COMMONAREA_MODE',
    'continuous__missingindicator_ELEVATORS_MODE',
    'continuous__missingindicator_ENTRANCES_MODE',
    'continuous__missingindicator_FLOORSMAX_MODE',
    'continuous__missingindicator_FLOORSMIN_MODE',
    'continuous__missingindicator_LANDAREA_MODE',
    'continuous__missingindicator_LIVINGAPARTMENTS_MODE',
    'continuous__missingindicator_LIVINGAREA_MODE',
    'continuous__missingindicator_NONLIVINGAPARTMENTS_MODE',
    'continuous__missingindicator_NONLIVINGAREA_MODE',
    'continuous__missingindicator_APARTMENTS_MEDI',
    'continuous__missingindicator_BASEMENTAREA_MEDI',
    'continuous__missingindicator_YEARS_BEGINEXPLUATATION_MEDI',
    'continuous__missingindicator_YEARS_BUILD_MEDI',
    'continuous__missingindicator_COMMONAREA_MEDI',
    'continuous__missingindicator_ELEVATORS_MEDI',
    'continuous__missingindicator_ENTRANCES_MEDI',
    'continuous__missingindicator_FLOORSMAX_MEDI',
    'continuous__missingindicator_FLOORSMIN_MEDI',
    'continuous__missingindicator_LANDAREA_MEDI',
    'continuous__missingindicator_LIVINGAPARTMENTS_MEDI',
    'continuous__missingindicator_LIVINGAREA_MEDI',
    'continuous__missingindicator_NONLIVINGAPARTMENTS_MEDI',
    'continuous__missingindicator_NONLIVINGAREA_MEDI',
    'continuous__missingindicator_TOTALAREA_MODE',
    'continuous__missingindicator_AMT_REQ_CREDIT_BUREAU_HOUR',
    'continuous__missingindicator_AMT_REQ_CREDIT_BUREAU_DAY',
    'continuous__missingindicator_AMT_REQ_CREDIT_BUREAU_WEEK',
    'continuous__missingindicator_AMT_REQ_CREDIT_BUREAU_MON',
    'continuous__missingindicator_AMT_REQ_CREDIT_BUREAU_QRT',
    'continuous__missingindicator_AMT_REQ_CREDIT_BUREAU_YEAR',
    'continuous__missingindicator_BUREAU_ACTIVE_SHARE',
    'continuous__missingindicator_BUREAU_ACTIVE_DEBT',
    'continuous__missingindicator_BUREAU_DAYS_OVERDUE_MAX',
    'continuous__missingindicator_BUREAU_DAYS_CREDIT_LATEST',
    'continuous__missingindicator_PREV_APPROVED_SHARE',
    'continuous__missingindicator_PREV_REFUSED_SHARE',
    'continuous__missingindicator_PREV_DAYS_DECISION_LATEST',
    'continuous__missingindicator_PREV_CREDIT_TO_REQUESTED_MEAN',
    'continuous__missingindicator_POS_LATE_MONTH_SHARE',
    'continuous__missingindicator_POS_DPD_MAX',
    'continuous__missingindicator_INST_LATE_SHARE',
    'continuous__missingindicator_INST_DAYS_LATE_MEAN',
    'continuous__missingindicator_INST_DAYS_LATE_MAX',
    'continuous__missingindicator_INST_UNDERPAID_SHARE',
    'continuous__missingindicator_INST_PAID_TO_DUE_MEAN',
    'card__CC_UTILISATION_LATEST',
    'card__CC_UTILISATION_MEAN',
    'card__CC_DRAWINGS_MEAN',
    'binary__FLAG_EMP_PHONE',
    'binary__FLAG_WORK_PHONE',
    'binary__FLAG_PHONE',
    'binary__REG_CITY_NOT_LIVE_CITY',
    'binary__REG_CITY_NOT_WORK_CITY',
    'binary__LIVE_CITY_NOT_WORK_CITY',
    'binary__FLAG_DOCUMENT_3',
    'binary__FLAG_DOCUMENT_6',
    'binary__FLAG_DOCUMENT_16',
    'binary__PREV_LATEST_REFUSED',
    'binary__HAS_HC_CREDIT_CARD',
    'categorical__NAME_CONTRACT_TYPE_Cash loans',
    'categorical__NAME_CONTRACT_TYPE_Revolving loans',
    'categorical__CODE_GENDER_F',
    'categorical__CODE_GENDER_M',
    'categorical__FLAG_OWN_CAR_N',
    'categorical__FLAG_OWN_CAR_Y',
    'categorical__NAME_INCOME_TYPE_Pensioner',
    'categorical__NAME_INCOME_TYPE_State servant',
    'categorical__NAME_INCOME_TYPE_Working',
    'categorical__NAME_EDUCATION_TYPE_Higher education',
    'categorical__NAME_EDUCATION_TYPE_Secondary / secondary special',
    'categorical__NAME_FAMILY_STATUS_Civil marriage',
    'categorical__NAME_FAMILY_STATUS_Married',
    'categorical__NAME_FAMILY_STATUS_Single / not married',
    'categorical__NAME_FAMILY_STATUS_Widow',
    'categorical__NAME_HOUSING_TYPE_House / apartment',
    'categorical__NAME_HOUSING_TYPE_Rented apartment',
    'categorical__NAME_HOUSING_TYPE_With parents',
    'categorical__OCCUPATION_TYPE_Accountants',
    'categorical__OCCUPATION_TYPE_Core staff',
    'categorical__OCCUPATION_TYPE_Drivers',
    'categorical__OCCUPATION_TYPE_High skill tech staff',
    'categorical__OCCUPATION_TYPE_Laborers',
    'categorical__OCCUPATION_TYPE_Low-skill Laborers',
    'categorical__OCCUPATION_TYPE_Managers',
    'categorical__OCCUPATION_TYPE_Missing',
    'categorical__OCCUPATION_TYPE_Sales staff',
    'categorical__OCCUPATION_TYPE_Security staff',
    'categorical__ORGANIZATION_TYPE_Agriculture',
    'categorical__ORGANIZATION_TYPE_Business Entity Type 3',
    'categorical__ORGANIZATION_TYPE_Construction',
    'categorical__ORGANIZATION_TYPE_Government',
    'categorical__ORGANIZATION_TYPE_Medicine',
    'categorical__ORGANIZATION_TYPE_Missing',
    'categorical__ORGANIZATION_TYPE_School',
    'categorical__ORGANIZATION_TYPE_Self-employed',
    'categorical__ORGANIZATION_TYPE_Transport: type 3',
    'categorical__FONDKAPREMONT_MODE_Missing',
    'categorical__FONDKAPREMONT_MODE_reg oper account',
    'categorical__HOUSETYPE_MODE_Missing',
    'categorical__HOUSETYPE_MODE_block of flats',
    'categorical__WALLSMATERIAL_MODE_Missing',
    'categorical__WALLSMATERIAL_MODE_Panel',
    'categorical__WALLSMATERIAL_MODE_Stone, brick',
    'categorical__EMERGENCYSTATE_MODE_Missing',
    'categorical__EMERGENCYSTATE_MODE_No',
]

In [6]:
def clean_placeholders(table):
    """Replace known placeholder values with NaN."""
    cleaned = table.copy()
    cleaned[EMPLOYMENT_COL] = cleaned[EMPLOYMENT_COL].replace(employed_sentinel, np.nan)
    cleaned[categorical_cols] = cleaned[categorical_cols].replace(PLACEHOLDER_TOKENS, np.nan)
    return cleaned


X_train = clean_placeholders(train_sample).drop(columns=[ID_COL, TARGET])
X_val = clean_placeholders(val_sample).drop(columns=[ID_COL, TARGET])
X_test = clean_placeholders(test_joined).drop(columns=ID_COL)
y_train = train_sample[TARGET]
y_val = val_sample[TARGET]

all_features = X_train.columns
numeric_features = X_train.select_dtypes(include="number").columns
n_values = X_train[numeric_features].nunique()

# Group the columns by the treatment they need
binary_cols = n_values[n_values == 2].index.tolist()
card_cols = [col for col in numeric_features if col.startswith("CC_") and col not in binary_cols]
continuous_features = [col for col in numeric_features if col not in binary_cols + card_cols]
categorical_features = [col for col in all_features if col not in numeric_features]


def build_preprocessor():
    columns = ColumnTransformer([
        ("continuous", Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", StandardScaler()),
        ]), continuous_features),
        ("card", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), card_cols),
        ("binary", SimpleImputer(strategy="most_frequent"), binary_cols),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ])
    return Pipeline([
        ("columns", columns),
        ("variance", VarianceThreshold()),
    ])

preprocessor = build_preprocessor()
train_candidates = preprocessor.fit_transform(X_train)
names = pd.Index(preprocessor.get_feature_names_out())
positions = names.get_indexer(ANDY_FEATURES_200)
assert len(set(ANDY_FEATURES_200)) == len(positions) == 200
if (positions < 0).any():
    missing = [name for name, pos in zip(ANDY_FEATURES_200, positions) if pos < 0]
    raise ValueError(f'Missing feature columns: {missing}')

X_train_ready = train_candidates[:, positions].astype(np.float32)
del train_candidates
X_val_ready = preprocessor.transform(X_val)[:, positions].astype(np.float32)
X_test_ready = preprocessor.transform(X_test)[:, positions].astype(np.float32)
matrices = X_train_ready, X_val_ready, X_test_ready
assert tuple(x.shape for x in matrices) == ((100_000, 200), (25_000, 200), (48_744, 200))
assert all(np.isfinite(x).all() for x in matrices)
del X_train, X_val, X_test, train_sample, val_sample, test_joined
gc.collect()
print('Model inputs:', *(x.shape for x in matrices))

Model inputs: (100000, 200) (25000, 200) (48744, 200)


## 4. Logistic regression and decision tree

Both feature sets use `C=0.01` for logistic regression and `max_depth=12`, `min_samples_leaf=400` for the tree. The baseline comes from rerunning our original notebook on the same applicant IDs and labels.

In [7]:
models = {
    'Logistic regression': LogisticRegression(C=0.01, solver='lbfgs', max_iter=2000, random_state=SEED),
    'Decision tree': DecisionTreeClassifier(max_depth=12, min_samples_leaf=400, random_state=SEED),
}
test_probabilities = {}
scores = []
for name, model in models.items():
    with threadpool_limits(limits=2):
        model.fit(X_train_ready, y_train)
        p_train = model.predict_proba(X_train_ready)[:, 1]
        p_val = model.predict_proba(X_val_ready)[:, 1]
        test_probabilities[name] = model.predict_proba(X_test_ready)[:, 1]
    scores.append((name, roc_auc_score(y_train, p_train), roc_auc_score(y_val, p_val)))

scores = pd.DataFrame(scores, columns=['Model', 'Train ROC-AUC', 'Validation ROC-AUC'])

baseline = pd.DataFrame([
    ('Logistic regression', 0.7747391253642197, 0.7671487031821086),
    ('Decision tree', 0.7626647520123961, 0.7135760640528861),
], columns=scores.columns)
comparison = pd.concat([baseline.assign(Features='Our 200'), scores.assign(Features='Andy 200')],
                       ignore_index=True)[['Features', *scores.columns]]
print(comparison.to_string(index=False, float_format=lambda x: f'{x:.6f}'))

Features               Model  Train ROC-AUC  Validation ROC-AUC
 Our 200 Logistic regression       0.774739            0.767149
 Our 200       Decision tree       0.762665            0.713576
Andy 200 Logistic regression       0.763962            0.760919
Andy 200       Decision tree       0.759607            0.713136


## 5. Export test probabilities

In [8]:
OUTPUT_DIR.mkdir(exist_ok=True)
pd.DataFrame({'feature': ANDY_FEATURES_200}).to_csv(OUTPUT_DIR / 'andy_features_200.csv', index=False)
submission_ids = pd.read_csv(DATA_DIR / 'sample_submission.csv')[ID]
order = test_ids.get_indexer(submission_ids)
assert len(order) == len(test_ids) == 48_744 and (order >= 0).all()
assert len(np.unique(order)) == len(order)

preview = pd.DataFrame({ID: submission_ids.head(3)})
for name, filename in [('Logistic regression', 'submission_andy_logreg.csv'),
                       ('Decision tree', 'submission_andy_tree.csv')]:
    p = test_probabilities[name][order]
    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
    submission = pd.DataFrame({ID: submission_ids, 'TARGET': p})
    submission.to_csv(OUTPUT_DIR / filename, index=False, float_format='%.12g')
    preview[name] = p[:3]
    print(f'{filename}: {len(submission):,} predictions')

print(preview.to_string(index=False, float_format=lambda x: f'{x:.6f}'))

submission_andy_logreg.csv: 48,744 predictions
submission_andy_tree.csv: 48,744 predictions
 SK_ID_CURR  Logistic regression  Decision tree
     100001             0.041057       0.051345
     100005             0.121228       0.035851
     100013             0.029018       0.012176
